In [1]:
from pathlib import Path
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUT_PATH = DATA_PROCESSED / "uss_english_turns_llm_satisfaction.parquet"
CACHE_PATH = DATA_PROCESSED / "llm_satisfaction_cache.jsonl"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_PATH:", OUT_PATH)
print("CACHE_PATH:", CACHE_PATH)


PROJECT_ROOT: C:\Users\User\Documents\retailmind-prototype
OUT_PATH: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_turns_llm_satisfaction.parquet
CACHE_PATH: C:\Users\User\Documents\retailmind-prototype\data\processed\llm_satisfaction_cache.jsonl


In [2]:
in_path = DATA_PROCESSED / "uss_english_issue_tagged.parquet"
df = pd.read_parquet(in_path)

print("Loaded:", in_path, df.shape)
df.head(2)


Loaded: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_issue_tagged.parquet (39102, 16)


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,dataset,low_satisfaction,entity_tag,issues,severity,reason
0,0,1,USER,What is the weather like on the March 4th?,INFORM,"3,3,3","[3, 3, 3]",False,3.0,INFORM,SGD,False,None,[],NONE,
1,0,2,SYSTEM,In which city should I look?,REQUEST,None,None,False,NaN,REQUEST,SGD,False,None,[],NONE,


In [3]:
df = df.sort_values(["dataset", "conv_id", "turn_id"]).copy()

df["system_text_only"] = df["text"].where(df["speaker"] == "SYSTEM")
df["last_system_text"] = (
    df.groupby(["dataset", "conv_id"])["system_text_only"]
      .ffill()
)

# last_system_text should be non-null for most USER turns after first bot message
df[["dataset","conv_id","turn_id","speaker","text","last_system_text"]].head(10)


,dataset,conv_id,turn_id,speaker,text,last_system_text
26666,CCPE,0,1,SYSTEM,Do you like movies like Thor?,Do you like movies like Thor?
26667,CCPE,0,2,USER,"No, I don't like Thor.",Do you like movies like Thor?
26668,CCPE,0,3,SYSTEM,Ok. What is it about this type of movie that y...,Ok. What is it about this type of movie that y...
26669,CCPE,0,4,USER,I don't like all the,Ok. What is it about this type of movie that y...
26670,CCPE,0,5,USER,Like the I don't know. Like is it voice acting?,Ok. What is it about this type of movie that y...
26671,CCPE,0,6,USER,in that controls the characters,Ok. What is it about this type of movie that y...
26672,CCPE,0,7,SYSTEM,Can you say a little more about that please?,Can you say a little more about that please?
26673,CCPE,0,8,USER,"Yeah, so I don't know I guess the acting was bad.",Can you say a little more about that please?
26674,CCPE,0,9,USER,I was wondering if it's voice acting that cont...,Can you say a little more about that please?
26675,CCPE,0,10,SYSTEM,Do you like movies like Star Wars?,Do you like movies like Star Wars?


In [4]:
mask_score = (df["speaker"] == "USER") & (df.get("is_overall", False) == False)
df_score = df[mask_score].copy()

print("Rows to score:", len(df_score))
df_score[["dataset","conv_id","turn_id","text","last_system_text"]].head(5)


Rows to score: 19193


,dataset,conv_id,turn_id,text,last_system_text
26667,CCPE,0,2,"No, I don't like Thor.",Do you like movies like Thor?
26669,CCPE,0,4,I don't like all the,Ok. What is it about this type of movie that y...
26670,CCPE,0,5,Like the I don't know. Like is it voice acting?,Ok. What is it about this type of movie that y...
26671,CCPE,0,6,in that controls the characters,Ok. What is it about this type of movie that y...
26673,CCPE,0,8,"Yeah, so I don't know I guess the acting was bad.",Can you say a little more about that please?


In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads .env in project root

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found. Ensure .env exists in project root and is loaded.")

client = OpenAI(api_key=api_key)

# Config
MODEL_SAT = "gpt-4o-mini"   # fast/cheap default; can switch to "gpt-4o" for potentially better quality
TEMPERATURE = 0
MAX_RETRIES = 3
SLEEP_BETWEEN_RETRIES = 1.0

print("Client ready. Model:", MODEL_SAT)


Client ready. Model: gpt-4o-mini


In [10]:
SYSTEM_PROMPT_SAT = """You are an evaluator for chatbot conversations.
Your task: given a user message and the assistant message that immediately preceded it, assign a satisfaction score from 1 to 5.

Scoring rubric:
5 = user is clearly satisfied / success / expresses thanks or confirms solved
4 = mostly satisfied / minor issues but positive
3 = neutral / unclear satisfaction / mixed
2 = dissatisfied / frustration / not solved
1 = very dissatisfied / angry / repeated failure / refuses system response

Return ONLY valid JSON with exactly these keys:
- "score": integer in {1,2,3,4,5}
- "reason": short string (max 25 words), explaining why
"""

def build_user_prompt(user_text: str, last_system_text: str) -> str:
    user_text = "" if user_text is None else str(user_text)
    last_system_text = "" if last_system_text is None else str(last_system_text)

    return (
        f'Assistant previous message:\n"{last_system_text}"\n\n'
        f'User message:\n"{user_text}"\n\n'
        "Output JSON now."
    )


In [11]:
def hash_key(dataset, conv_id, turn_id, user_text, last_system_text, model_name):
    s = f"{dataset}|{conv_id}|{turn_id}|{model_name}|{user_text}|{last_system_text}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def load_cache_jsonl(path: Path):
    cache = {}
    if not path.exists():
        return cache
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            cache[obj["key"]] = obj
    return cache

def append_cache_jsonl(path: Path, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

cache = load_cache_jsonl(CACHE_PATH)
print("Cache entries loaded:", len(cache))


Cache entries loaded: 0


In [12]:
def call_llm_satisfaction(user_text: str, last_system_text: str) -> dict:
    prompt = build_user_prompt(user_text, last_system_text)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_SAT,
                temperature=TEMPERATURE,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_SAT},
                    {"role": "user", "content": prompt},
                ],
            )
            content = resp.choices[0].message.content

            # Parse JSON strictly
            data = json.loads(content)

            # Validate schema
            if "score" not in data or "reason" not in data:
                raise ValueError(f"Missing keys in response: {data}")
            if not isinstance(data["score"], int) or data["score"] not in [1,2,3,4,5]:
                raise ValueError(f"Invalid score: {data['score']}")
            if not isinstance(data["reason"], str):
                raise ValueError("Reason must be a string")

            return {"score": data["score"], "reason": data["reason"]}

        except Exception as e:
            if attempt == MAX_RETRIES:
                # Safe fallback
                return {"score": 3, "reason": f"LLM_ERROR: {type(e).__name__}"}
            time.sleep(SLEEP_BETWEEN_RETRIES * attempt)


In [13]:
df_score_small = df_score.head(30).copy()

rows = []
for _, r in df_score_small.iterrows():
    key = hash_key(r["dataset"], r["conv_id"], r["turn_id"], r["text"], r["last_system_text"], MODEL_SAT)

    if key in cache:
        out = cache[key]
        score = out["score"]
        reason = out["reason"]
        source = "cache"
    else:
        res = call_llm_satisfaction(r["text"], r["last_system_text"])
        score = res["score"]
        reason = res["reason"]
        source = "llm"

        obj = {
            "key": key,
            "dataset": r["dataset"],
            "conv_id": int(r["conv_id"]),
            "turn_id": int(r["turn_id"]),
            "model": MODEL_SAT,
            "score": int(score),
            "reason": reason,
        }
        append_cache_jsonl(CACHE_PATH, obj)
        cache[key] = obj

    rows.append({
        "dataset": r["dataset"],
        "conv_id": r["conv_id"],
        "turn_id": r["turn_id"],
        "score": score,
        "reason": reason,
        "source": source,
    })

df_test = pd.DataFrame(rows)
print(df_test["score"].value_counts().sort_index())
df_test.head(10)


score
2     6
3    10
4     7
5     7
Name: count, dtype: int64


,dataset,conv_id,turn_id,score,reason,source
0,CCPE,0,2,2,User expresses clear dissatisfaction with the ...,llm
1,CCPE,0,4,3,"User's response is incomplete, indicating unce...",llm
2,CCPE,0,5,3,User's response is unclear and shows uncertain...,llm
3,CCPE,0,6,3,User's response is unclear and does not expres...,llm
4,CCPE,0,8,3,User provides feedback but is unclear about sa...,llm
5,CCPE,0,9,2,User expresses frustration and feels the assis...,llm
6,CCPE,0,11,3,"User expresses a neutral stance, indicating so...",llm
7,CCPE,0,12,5,User expresses clear satisfaction by confirmin...,llm
8,CCPE,0,14,4,"User expresses a positive aspect of the movie,...",llm
9,CCPE,0,15,5,User expresses satisfaction with the movie's a...,llm


In [15]:
MAX_TURNS_TO_SCORE = 3000     # demo-friendly; change later to expand
RANDOM_SEED = 42

# Keep the same df_score definition, but limit it here
df_score_run = df_score.sample(
    n=min(MAX_TURNS_TO_SCORE, len(df_score)),
    random_state=RANDOM_SEED
).copy()

print("Total USER turns available:", len(df_score))
print("USER turns selected to score now:", len(df_score_run))


Total USER turns available: 19193
USER turns selected to score now: 3000


In [16]:
from tqdm import tqdm

results = []
n_total = len(df_score_run)

for i, r in tqdm(enumerate(df_score_run.itertuples(index=False), start=1)):
    key = hash_key(r.dataset, r.conv_id, r.turn_id, r.text, r.last_system_text, MODEL_SAT)

    if key in cache:
        score = cache[key]["score"]
        reason = cache[key]["reason"]
        source = "cache"
    else:
        res = call_llm_satisfaction(r.text, r.last_system_text)
        score = res["score"]
        reason = res["reason"]
        source = "llm"

        obj = {
            "key": key,
            "dataset": r.dataset,
            "conv_id": int(r.conv_id),
            "turn_id": int(r.turn_id),
            "model": MODEL_SAT,
            "score": int(score),
            "reason": reason,
        }
        append_cache_jsonl(CACHE_PATH, obj)
        cache[key] = obj

    results.append((r.dataset, r.conv_id, r.turn_id, int(score), reason, source))

    if i % 200 == 0:
        print(f"Scored {i}/{n_total} (new run subset)")

df_scores = pd.DataFrame(
    results,
    columns=["dataset", "conv_id", "turn_id", "llm_satisfaction_score", "llm_satisfaction_reason", "source"]
)

print("Scored rows in this run:", df_scores.shape)
print(df_scores["llm_satisfaction_score"].value_counts().sort_index())
df_scores.head(10)


200it [03:02,  1.29it/s]

Scored 200/3000 (new run subset)


400it [05:54,  1.13it/s]

Scored 400/3000 (new run subset)


600it [08:46,  1.15it/s]

...

Scored 2800/3000 (new run subset)


3000it [1:43:05,  2.06s/it]

Scored 3000/3000 (new run subset)


Scored rows in this run: (3000, 6)
llm_satisfaction_score
1     125
2     381
3    1075
4     317
5    1102
Name: count, dtype: int64


,dataset,conv_id,turn_id,llm_satisfaction_score,llm_satisfaction_reason,source
0,SGD,336,27,5,User confirms the assistant's question positiv...,llm
1,SGD,325,1,1,User is frustrated due to the assistant's unhe...,llm
2,CCPE,400,13,2,User is confused and dissatisfied with the ass...,llm
3,SGD,41,1,1,User is frustrated due to the assistant's unhe...,llm
4,SGD,258,17,2,"User declined the offer, indicating dissatisfa...",llm
5,SGD,459,3,5,"User clearly specifies their preference, indic...",llm
6,SGD,769,11,5,User confirms the appointment and requests add...,llm
7,SGD,298,31,5,"User expresses gratitude, indicating satisfact...",llm
8,CCPE,413,8,3,User expresses a neutral opinion about a movie...,llm
9,SGD,821,19,5,User confirms satisfaction by agreeing to the ...,llm


In [17]:
df_out = df.copy()

# Merge the subset scores into the full dataset
df_out = df_out.merge(
    df_scores[["dataset","conv_id","turn_id","llm_satisfaction_score","llm_satisfaction_reason"]],
    on=["dataset","conv_id","turn_id"],
    how="left",
)

# Flag low satisfaction only where we have a score
df_out["low_satisfaction_llm"] = df_out["llm_satisfaction_score"].apply(
    lambda x: (x < 3) if pd.notna(x) else False
)

# Ensure non-USER turns don't carry scores
df_out.loc[df_out["speaker"] != "USER", ["llm_satisfaction_score", "llm_satisfaction_reason", "low_satisfaction_llm"]] = [np.nan, None, False]

print("USER scored coverage (this run):",
      df_out.loc[df_out["speaker"]=="USER", "llm_satisfaction_score"].notna().mean())

print("Low-satisfaction rate among SCORED USER turns:",
      df_out.loc[(df_out["speaker"]=="USER") & (df_out["llm_satisfaction_score"].notna()), "low_satisfaction_llm"].mean())



USER scored coverage (this run): 0.14497656212245688
Low-satisfaction rate among SCORED USER turns: 0.16866666666666666


In [18]:
OUT_PATH_SUBSET = DATA_PROCESSED / "uss_english_turns_llm_satisfaction_subset.parquet"

df_out.to_parquet(OUT_PATH_SUBSET, index=False)
print("Saved:", OUT_PATH_SUBSET)
print("Rows:", len(df_out))


Saved: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_turns_llm_satisfaction_subset.parquet
Rows: 39102


In [19]:
print("Columns present:", [c for c in ["llm_satisfaction_score","llm_satisfaction_reason","low_satisfaction_llm"] if c in df_out.columns])

print("\nScore distribution (scored USER turns):")
display(
    df_out.loc[(df_out["speaker"]=="USER") & df_out["llm_satisfaction_score"].notna(), "llm_satisfaction_score"]
    .value_counts().sort_index()
)

print("\nExample low-satisfaction rows:")
cols_show = ["dataset","conv_id","turn_id","text","last_system_text","llm_satisfaction_score","llm_satisfaction_reason","low_satisfaction_llm"]
display(df_out.loc[(df_out["speaker"]=="USER") & (df_out["low_satisfaction_llm"]==True), cols_show].head(8))


Columns present: ['llm_satisfaction_score', 'llm_satisfaction_reason', 'low_satisfaction_llm']

Score distribution (scored USER turns):


llm_satisfaction_score
1.0     125
2.0     381
3.0    1075
4.0     317
5.0    1102
Name: count, dtype: int64


Example low-satisfaction rows:


,dataset,conv_id,turn_id,text,last_system_text,llm_satisfaction_score,llm_satisfaction_reason,low_satisfaction_llm
91,CCPE,2,33,"Oh, I dislike it. It's I don't know if it's of...",Do you like or dislike the movie Aliens?,2.0,User expresses strong dislike and frustration ...,True
351,CCPE,13,22,"So, I would say I'm kind of in the middle on t...",did you like it?,2.0,"User expresses a negative opinion, indicating ...",True
464,CCPE,18,10,I found the movie to be just too over the top ...,What about that kind of movie didn't you like?...,2.0,User expresses dissatisfaction with the movie ...,True
489,CCPE,19,16,It's just boring boring to me.,what do you dislike about romance movies?,2.0,User expresses dissatisfaction and finds roman...,True
637,CCPE,25,19,"Nope, haven't seen that.",How about the Incredibles 2,2.0,User expresses dissatisfaction by rejecting th...,True
710,CCPE,28,8,Don't really know what that is.,do you like movies like Christopher Robin?,2.0,User expresses confusion and lack of knowledge...,True
729,CCPE,29,12,They annoy me. I like movies that have a theme...,why don't you like musicals?,2.0,"User expresses frustration with musicals, indi...",True
754,CCPE,30,16,I just don't really care too much for the subj...,why?,2.0,"User expresses disinterest, indicating dissati...",True
